In [1]:
# Licensed to the Apache Software Foundation (ASF) under one
# or more contributor license agreements.  See the NOTICE file
# distributed with this work for additional information
# regarding copyright ownership.  The ASF licenses this file
# to you under the Apache License, Version 2.0 (the
# "License"); you may not use this file except in compliance
# with the License.  You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing,
# software distributed under the License is distributed on an
# "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
# KIND, either express or implied.  See the License for the
# specific language governing permissions and limitations
# under the License.

import os,glob
import logging

import numpy as np
from standalone_ms_cost_model import load_tir
import pandas as pd
from tvm import relay
from tvm.relay.backend import Executor
from tvm import meta_schedule as ms
from tvm.driver import tvmc
from tvm.meta_schedule.runner import EvaluatorConfig
from tvm.meta_schedule.logging import get_logger
from tvm.contrib.micro.meta_schedule.local_builder_micro import get_local_builder_micro
from tvm.contrib.micro.meta_schedule.rpc_runner_micro import get_rpc_runner_micro
from tvm.rpc import connect_tracker
from model_info import get_model_info


logging.basicConfig(level=logging.DEBUG)
get_logger("xgb_model").setLevel(logging.DEBUG)

# DIR = Path(__file__).parent.resolve()
# BASE_DIR = DIR.parent

GCC_PREFIX = os.environ.get("GCC_PREFIX", "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/deps/install/riscv_gcc_rv32")
GCC_NAME = os.environ.get("GCC_NAME", "riscv32-unknown-elf")
LLVM_DIR = os.environ.get("LLVM_DIR", "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/deps/install/llvm")
ETISS_TEMPLATE =  "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/deps/src/microtvm-etiss-template"
ETISS_SCRIPT = os.environ.get("ETISS_SCRIPT", "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/deps/install/etiss/bin/run_helper.sh")
PLATFORM = os.path.join(ETISS_TEMPLATE, "template_project")


def load_model(model):
    def _load_model(path, shape_dict):
        model = tvmc.load(
            str(path),
            shape_dict=shape_dict,
        )
        mod = model.mod
        params = model.params
        return mod, params

    model_info = get_model_info(model)
    shape_dict = {t.name: t.shape for t in model_info.in_tensors}
    assert len(model_info.in_tensors) == 1
    input_name, input_shape = list(shape_dict.items())[0]
    input_dtype = model_info.in_tensors[0].dtype
    data_sample = np.random.rand(*input_shape).astype(input_dtype)
    mod, params = _load_model(model, shape_dict)
    return mod, params, input_name, input_shape, input_dtype, data_sample


def get_tuning_config():
    def _get_sch_rules():
        structure = "SR"
        return [
            ms.schedule_rule.ApplyCustomRule(),
            ms.schedule_rule.InlineConstantScalars(),
            ms.schedule_rule.AutoInline(
                into_producer=False,
                into_consumer=True,
                inline_const_tensor=True,
                disallow_if_then_else=True,
                require_injective=True,
                require_ordered=True,
                disallow_op=["tir.exp"],
            ),
            ms.schedule_rule.MultiLevelTiling(
                structure="SSRSRS",
                tile_binds=None,
                max_innermost_factor=64,
                vector_load_lens=None,
                reuse_read=None,
                reuse_write=ms.schedule_rule.ReuseType(
                    req="may",
                    levels=[1, 2],
                    scope="global",
                ),
            ),
            ms.schedule_rule.ParallelizeVectorizeUnroll(
                max_jobs_per_core=-1,  # disable parallelize
                max_vectorize_extent=-1,  # disable vectorize
                unroll_max_steps=[0, 2, 4, 8, 16, 32, 64],
                unroll_explicit=True,
                # unroll_explicit=False,
            ),
            ms.schedule_rule.RandomComputeLocation(),
        ]

    def _get_postprocs():
        return [
            ms.postproc.DisallowDynamicLoop(),
            ms.postproc.RewriteParallelVectorizeUnroll(),
            ms.postproc.RewriteReductionBlock(),
        ]

    def _get_mutator_probs():
        return {
            ms.mutator.MutateTileSize(): 0.9,
            ms.mutator.MutateComputeLocation(): 0.05,
            ms.mutator.MutateUnroll(): 0.03,
            # ms.mutator.Parallel(): 0.02,
        }

    sch_rules = _get_sch_rules()
    postprocs = _get_postprocs()
    mutator_probs = _get_mutator_probs()
    return sch_rules, postprocs, mutator_probs


def _schedule_dummy():

    def schedule_fn(sch, block=None) -> bool:
        return True

    return schedule_fn


ALTER_OP = True
TOOLCHAIN = "gcc"
TARGET = "c -num-cores 1"
NUM_TRIALS_PER_ITER, MAX_TRIALS_PER_TASK, MAX_TRIALS_GLOBAL = (5, 50, 1000000)
TASK_FILTER = [0, 1]
MODULE_EQUALITY = "ignore-ndarray"
TRANSFORM_LAYOUT = False

OPTIONS = {
    "verbose": True,
    "quiet": True,
    "gcc_prefix": str(GCC_PREFIX),
    "gcc_name": GCC_NAME,
    "llvm_dir": str(LLVM_DIR),
    "etiss_script": str(ETISS_SCRIPT),
    "etiss_args": "",
    "arch": "rv32gc_zicsr_zifencei",
    "abi": "ilp32d",
    "cpu_arch": "RV32IMACFD",
    "cpu_freq": 100000000,
    "toolchain": TOOLCHAIN,
}

MS_DISPATCH = 1  # silent?
# MS_DISPATCH = 2  # verbose
# MS_DISPATCH = ?  # error
SKIP_TUNING = False

In [7]:
# from tvm.contrib.micro.meta_schedule.rpc_runner_micro import get_rpc_runner_micro
# from tvm.meta_schedule.runner import EvaluatorConfig
# from tvm.rpc import connect_tracker
# # Example platform and options (adjust as needed for your setup)
# platform = "crt"
# options = {}

# with get_rpc_runner_micro(
#     platform=platform,
#     options=options,
#     evaluator_config=EvaluatorConfig(number=1, repeat=1, min_repeat_ms=100),
#     tracker_host="127.0.0.1",
#     tracker_port=9190,
#     serial_numbers=["$local$device"],  # or your actual device key(s)
# ) as runner:
#     print("RPC runner and server started successfully!")
#     # Connect to the tracker and print summary
#     tracker = connect_tracker("127.0.0.1", 9190)
#     print("Tracker summary:\n", tracker.summary())

In [3]:
def get_report_from_session(session):
    home = os.getenv("MLONMCU_HOME")
    report= glob.glob(os.path.join(home,"temp/sessions",str(session),"report.csv"))[0]
    print(f"Report file found: {report}")
    df = pd.read_csv(report, sep=",")
    return df

# df = get_report_from_session(1054)


In [4]:


def get_sublayer_samples_from_session(session_id):
    path = f"/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/temp/sessions/{session_id}"
    report = get_report_from_session(session_id)
    data = []
    samples_rom=[]
    samples_inst=[]
    temp = {}
    # path = "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/temp/sessions/1054"
    for id,row in report.dropna(axis=0,subset=['Sub']).iterrows():
        temp["dir_path"] = os.path.join(path,"runs",str(row["Run"]),"sub",row["Sub"])
        # if glob.glob(os.path.join(temp["dir_path"],"default.tir")):
        temp["tir_files"] = glob.glob(os.path.join(temp["dir_path"],"default.tir"))[0]
        temp["Rom Code"] = row['ROM code']
        temp['Run Cycles'] = row['Run Cycles']
        
        mods = []
        # for tir_file in temp["tir_files"]:
        mods.append(load_tir(temp["tir_files"]))
        temp["mods"] = mods
        # print(temp)
        data.append(temp.copy())
    return data

# data = get_sublayer_samples_from_session(1054)  
# data



In [5]:
mods = load_tir('/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/temp/sessions/1054/runs/0/sub/layer8/default.tir')

## Test with RPC Runtime

In [6]:
link_params = True
runtime = relay.backend.Runtime("crt", {"system-lib": True})
executor = Executor("aot", {"link-params": link_params})
mods = [mod.with_attr("executor", executor) for mod in mods]


In [7]:
builder = get_local_builder_micro()

INFO:tvm.meta_schedule.builder.local_builder:LocalBuilder: max_workers = 4


In [1]:
# from tvm.target import Target
# TARGET = "c -num-cores 1"
# targ = Target(TARGET)
import os
def get_all_tflite_files(directory):
    tflite_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".tflite"):
                tflite_files.append(os.path.join(root, file))
    return tflite_files
model_path= "/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models"
    # MODELS = ["/mobilenet_v1_1_0_224_quant/mobilenet_v1_1_0_224_quant.tflite","/lstm2/lstm2.tflite",
    #           "/cifar10/cifar10.tflite",""]
    
    
    

    # Example usage:
tflite_files = get_all_tflite_files(model_path)

In [4]:
pass_config = {
        "tir.disable_vectorize": True,'tir.max_stack_alloca':1024
    }    
pass_config['tir.max_stack_alloca']=32
pass_config

{'tir.disable_vectorize': True, 'tir.max_stack_alloca': 32}

In [ ]:
from pprint import pprint
pprint(tflite_files)
MODEL=[ '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/aww/aww.tflite',
       '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/MobileNetV2/MobileNet_V2.tflite',
       '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/lstm2/lstm2.tflite',
        '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/vww/vww.tflite',
         '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/toycar/toycar.tflite',
          '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/resnet/resnet.tflite',
           '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/magic_wand/magic_wand.tflite']

['/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/2sort/SMD_Ori_tflite.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/MobileNetV2/MobileNetV2.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/MobileNetV2/MobileNet_V2.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/aww/aww.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/bigsine_quant/bigsine_quant.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/catdog/CNN_2L_model_quantized.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/catdog/CNN_full_quantized.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/catdog/CNN_full_quantized_70_.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/catdog/CNN_model.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/catdog/catdog.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/catdog2/catdog2.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/cifar10/cifar10.tflite',
 '/nfs/TUEIEDAscratch/ge85zic/mlonmcu_env/models/

In [11]:
# with ms.Profiler() as profiler:
evaluator_config = EvaluatorConfig(
                number=1,
                repeat=1,
                min_repeat_ms=0,
                enable_cpu_cache_flush=False,
            )
with get_rpc_runner_micro(
                platform=PLATFORM, options=OPTIONS, session_timeout_sec=120, evaluator_config=evaluator_config,
                # serial_numbers=["micro"] * micro_rpc_workers,
                tracker_host="127.0.0.1",
                tracker_port=9190,
                # max_workers=micro_rpc_workers,
                rpc_timeout_sec=10,

            ) as runner:
    print("RPC runner and server started successfully!")
    # Connect to the tracker and print summary
    tracker = connect_tracker("127.0.0.1", 9190)
    print("Tracker summary:\n", tracker.summary())


INFO:tvm.contrib.micro.meta_schedule.rpc_runner_micro:RPCRunner: max_workers = 4


RPC runner and server started successfully!
Tracker summary:
 {'queue_info': {'$local$device': {'free': 1, 'pending': 0}}, 'server_info': [{'key': 'server:$local$device', 'addr': ['127.0.0.1', 9191]}]}
